# MNIST with TensorFlow / Nhập môn nhận dạng chữ số

Tôi luyện mô hình dense `784 → 128 → 10`, dropout, logits và cross-entropy.
**Input:** ảnh MNIST 28×28. **Output:** xác suất 10 chữ số và accuracy trên test.

Practice a dense network, dropout, logits and cross-entropy.
Run all cells with Python 3.12 and TensorFlow; use the [run guide](../README.md).
Full official dataset: **54,000 train / 6,000 validation / 10,000 test**, seed 42, 5 epochs.

## Nguồn / Attribution

Bài học Colab cá nhân dựa trên [TensorFlow beginner quickstart](https://www.tensorflow.org/tutorials/quickstart/beginner).
Tôi bổ sung seed, tập validation, chuẩn hóa float32 và kiểm tra xác suất đầu ra.
This learning adaptation adds a separate validation split and repeatable execution settings.

In [1]:
import os
os.environ.setdefault('TF_ENABLE_ONEDNN_OPTS', '0')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

SEED = 42
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()
# Hai luồng CPU, phù hợp máy nhỏ / Bound CPU threads for small machines.
tf.config.threading.set_intra_op_parallelism_threads(2)
tf.config.threading.set_inter_op_parallelism_threads(2)
print('TensorFlow:', tf.__version__, '| Keras:', tf.keras.__version__)

TensorFlow: 2.21.0 | Keras: 3.15.1


In [2]:
from pathlib import Path
import hashlib

mnist = tf.keras.datasets.mnist
(x_all, y_all), (x_test, y_test) = mnist.load_data()
mnist_path = Path(os.environ.get('KERAS_HOME', str(Path.home() / '.keras'))) / 'datasets/mnist.npz'
mnist_sha256 = hashlib.sha256(mnist_path.read_bytes()).hexdigest()
assert mnist_sha256 == '731c5ac602752760c8e48fbffcf8c3b850d9dc2a2aedcf2cc48468fc17b673d1'
assert x_all.shape == (60000, 28, 28) and x_test.shape == (10000, 28, 28)
x_all = x_all.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train, x_valid, y_train, y_valid = train_test_split(
    x_all, y_all, test_size=0.1, stratify=y_all, random_state=SEED
)
del x_all, y_all
BATCH_SIZE = 128
options = tf.data.Options()
options.threading.private_threadpool_size = 2
options.threading.max_intra_op_parallelism = 1

def batches(x, y, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        dataset = dataset.shuffle(len(y), seed=SEED, reshuffle_each_iteration=True)
    return dataset.batch(BATCH_SIZE).with_options(options).prefetch(1)

train_ds = batches(x_train, y_train, training=True)
valid_ds = batches(x_valid, y_valid)
test_ds = batches(x_test, y_test)
print('Train/validation/test:', len(y_train), len(y_valid), len(y_test))
print('MNIST SHA-256:', mnist_sha256)

Train/validation/test: 54000 6000 10000
MNIST SHA-256: 731c5ac602752760c8e48fbffcf8c3b850d9dc2a2aedcf2cc48468fc17b673d1


In [3]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10),  # logits: chưa phải xác suất / unnormalized outputs
])
model.summary()

Model: "sequential"


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘


 Total params: 101,770 (397.54 KB)


 Trainable params: 101,770 (397.54 KB)


 Non-trainable params: 0 (0.00 B)


In [4]:
predictions = model(x_train[:1], training=False).numpy()
probabilities_before_training = tf.nn.softmax(predictions).numpy()
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
print('Initial probabilities:', probabilities_before_training)
print('Initial example loss:', float(loss_fn(y_train[:1], predictions)))

Initial probabilities: [[0.13534747 0.11992209 0.07234254 0.07572695 0.14007129 0.0540201
  0.08906726 0.11814984 0.12312476 0.07222765]]
Initial example loss: 2.6263427734375


In [5]:
model.compile(optimizer='adam', loss=loss_fn, metrics=['accuracy'])
history = model.fit(train_ds, validation_data=valid_ds, epochs=5, shuffle=False, verbose=2)

Epoch 1/5


422/422 - 2s - 4ms/step - accuracy: 0.8796 - loss: 0.4224 - val_accuracy: 0.9372 - val_loss: 0.2237


Epoch 2/5


422/422 - 1s - 2ms/step - accuracy: 0.9417 - loss: 0.2030 - val_accuracy: 0.9542 - val_loss: 0.1627


Epoch 3/5


422/422 - 1s - 2ms/step - accuracy: 0.9553 - loss: 0.1530 - val_accuracy: 0.9612 - val_loss: 0.1335


Epoch 4/5


422/422 - 1s - 2ms/step - accuracy: 0.9628 - loss: 0.1249 - val_accuracy: 0.9645 - val_loss: 0.1173


Epoch 5/5


422/422 - 1s - 2ms/step - accuracy: 0.9681 - loss: 0.1077 - val_accuracy: 0.9682 - val_loss: 0.1054


In [6]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
probability_model = tf.keras.Sequential([model, tf.keras.layers.Softmax()])
probabilities = probability_model(x_test[:5], training=False).numpy()
assert probabilities.shape == (5, 10)
assert np.isfinite(test_loss) and 0 <= test_accuracy <= 1
assert np.isfinite(probabilities).all() and (probabilities >= 0).all()
np.testing.assert_allclose(probabilities.sum(axis=1), 1.0, atol=1e-6)
LAB_RESULT = {
    'test_loss': float(test_loss), 'test_accuracy': float(test_accuracy),
    'train_rows': len(y_train), 'validation_rows': len(y_valid), 'test_rows': len(y_test),
    'epochs': len(history.history['loss']), 'batch_size': BATCH_SIZE,
    'last_validation_accuracy': float(history.history['val_accuracy'][-1]),
    'data_sha256': mnist_sha256,
}
print(LAB_RESULT)
print('Predicted digits:', probabilities.argmax(axis=1), '| Actual:', y_test[:5])

{'test_loss': 0.09480799734592438, 'test_accuracy': 0.9714000225067139, 'train_rows': 54000, 'validation_rows': 6000, 'test_rows': 10000, 'epochs': 5, 'batch_size': 128, 'last_validation_accuracy': 0.9681666493415833, 'data_sha256': '731c5ac602752760c8e48fbffcf8c3b850d9dc2a2aedcf2cc48468fc17b673d1'}
Predicted digits: [7 2 1 0 4] | Actual: [7 2 1 0 4]


## Điều tôi rút ra / What I learned

Lớp cuối trả logits nên loss cần `from_logits=True`; softmax chỉ dùng để đọc xác suất.
Dropout hoạt động khi huấn luyện, tắt khi đánh giá. Tôi dùng validation để quan sát
quá trình học và giữ test cho đánh giá cuối. Đây là bài phân loại MNIST công khai;
ảnh chữ số từ camera cần được đánh giá riêng.

Logits, loss configuration and inference probabilities must agree. A good MNIST
result alone does not measure performance on new camera images.